In [1]:
import numpy as np
house = 0
noise = [0.1,0.25,0.5,1,1.75,2.75,4]
buildings = [0,1,3,4,5,8,9,10,11,12,13,16,17,20,21,22,23,24,25,26]
hours = 24
price = 'Realistic'

timings = {}

In [2]:
import torch
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error as mape
import src.data.dataprep as prep
import src.data.featurisation as features

def torch_py(torch_tensor):
    return torch_tensor.cpu().detach().numpy().flatten()

def rescale(values, scaler):
    rescaled_values = values * (scaler[1] - scaler[0]) + scaler[0]   
    return rescaled_values

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
mapes = {}

for building in buildings:
    mape_building = []
    for n in noise:
        # Import the base data and resample it from 5 minutes to hourly
        nl_data = prep.dutch_data('../data/Dutchdata_clean/building_' + str(building) + '.parquet', 'h', price=price)
        featurisation = features.Featurisation(nl_data)
        nl_data = featurisation.cyclic_features(yearly=False)[0]

        load_fcst_data = np.load('../data/load_fcsts/building_' + str(building) + '_' + str(n) + '_noise_forecast.npy')
        nl_data['load_fcst'] = load_fcst_data

        mape_building.append(mape(nl_data['load'], nl_data['load_fcst']))
    mapes[building] = mape_building